# Delta Lake Basics — 08: Change Data Feed (CDF)

## What you will learn
Change Data Feed lets you track every row-level change to a Delta table
— inserts, updates, and deletes — and read them as a changelog.

In this notebook:
1. Enabling CDF on a table
2. Reading changes — `_change_type` column
3. Update pre-image vs post-image
4. Incremental pipeline pattern — checkpoint-based processing
5. CDF for downstream replication
6. CDF limitations and best practices


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

In [ ]:
TABLE = f"{DATA_DIR}/08_cdf"
shutil.rmtree(TABLE, ignore_errors=True)

# Create table with CDF enabled
df.write.format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .save(TABLE)

# Verify CDF is enabled
detail = DeltaTable.forPath(spark, TABLE).detail().collect()[0]
print(f"CDF enabled: {detail['properties'].get('delta.enableChangeDataFeed', 'false')}")
print(f"Initial rows: {spark.read.format('delta').load(TABLE).count():,}")

v0 = DeltaTable.forPath(spark, TABLE).history(1).collect()[0]["version"]
print(f"Baseline version: {v0}")

## Step 1 — Make Changes and Read CDF


In [ ]:
# Make changes — all will be tracked
# 1. Update some rows
spark.sql(f"""
    UPDATE delta.`{TABLE}`
    SET status = 'shipped'
    WHERE status = 'confirmed' AND order_id % 5 = 0
""")

# 2. Delete some rows
spark.sql(f"""
    DELETE FROM delta.`{TABLE}`
    WHERE status = 'cancelled' AND price < 30
""")

# 3. Insert new rows
new_orders = df.limit(200) \
               .withColumn("order_id", F.col("order_id") + 200000)
new_orders.write.format("delta").mode("append").save(TABLE)

v_latest = DeltaTable.forPath(spark, TABLE).history(1).collect()[0]["version"]
print(f"Latest version: {v_latest}")
print()

# Read all changes since baseline
changes = (
    spark.read.format("delta")
         .option("readChangeFeed",   "true")
         .option("startingVersion",  v0 + 1)
         .load(TABLE)
)

print("Change summary:")
changes.groupBy("_change_type").count().orderBy("_change_type").show()

## Step 2 — Understanding Change Types

CDF produces four change types:

| `_change_type` | Meaning |
|---|---|
| `insert` | New row added |
| `update_preimage` | Row BEFORE update (old values) |
| `update_postimage` | Row AFTER update (new values) |
| `delete` | Row removed |


In [ ]:
# Inspect each change type
print("INSERT rows (new orders):")
changes.filter(F.col("_change_type") == "insert") \
       .select("order_id","status","_change_type","_commit_version") \
       .show(5)

print("UPDATE pairs (before + after):")
changes.filter(F.col("_change_type").isin("update_preimage","update_postimage")) \
       .select("order_id","status","_change_type","_commit_version") \
       .orderBy("order_id","_change_type") \
       .show(8)

print("DELETE rows (removed orders):")
changes.filter(F.col("_change_type") == "delete") \
       .select("order_id","status","price","_change_type","_commit_version") \
       .show(5)

## Step 3 — Incremental Pipeline Pattern

The most important CDF use case: process ONLY changed rows,
not the entire table, on every pipeline run.


In [ ]:
# Simulate incremental pipeline with checkpoint
CHECKPOINT_FILE = f"{DATA_DIR}/08_checkpoint.txt"

def get_checkpoint():
    try:
        return int(pathlib.Path(CHECKPOINT_FILE).read_text().strip())
    except FileNotFoundError:
        return 0

def save_checkpoint(version):
    pathlib.Path(CHECKPOINT_FILE).write_text(str(version))

def process_incremental():
    """Process only changes since last checkpoint."""
    last_version = get_checkpoint()
    latest_version = DeltaTable.forPath(spark, TABLE).history(1).collect()[0]["version"]

    if last_version >= latest_version:
        print("No new changes to process.")
        return 0

    changes = (
        spark.read.format("delta")
             .option("readChangeFeed", "true")
             .option("startingVersion", last_version + 1)
             .option("endingVersion",   latest_version)
             .load(TABLE)
             .filter(F.col("_change_type") != "update_preimage")  # skip before-images
    )

    n = changes.count()
    print(f"Processing {n} changes (v{last_version+1} → v{latest_version})")
    print(f"  Inserts: {changes.filter(F.col('_change_type')=='insert').count()}")
    print(f"  Updates: {changes.filter(F.col('_change_type')=='update_postimage').count()}")
    print(f"  Deletes: {changes.filter(F.col('_change_type')=='delete').count()}")

    save_checkpoint(latest_version)
    return n

print("=== Run 1: first incremental run ===")
process_incremental()

# Add more changes
spark.sql(f"UPDATE delta.`{TABLE}` SET status='delivered' WHERE status='shipped' LIMIT 100")
spark.sql(f"INSERT INTO delta.`{TABLE}` SELECT * FROM delta.`{TABLE}` WHERE false")  # noop

print("\n=== Run 2: new changes ===")
process_incremental()

print("\n=== Run 3: no changes ===")
process_incremental()

## Summary

```python
# Enable CDF at creation
df.write.format("delta")
        .option("delta.enableChangeDataFeed", "true")
        .mode("overwrite").save(path)

# Enable CDF on existing table
spark.sql(f"ALTER TABLE delta.`/path` SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

# Read changes (version range)
spark.read.format("delta")
     .option("readChangeFeed", "true")
     .option("startingVersion", 5)
     .option("endingVersion", 10)
     .load(path)

# Read changes (timestamp range)
spark.read.format("delta")
     .option("readChangeFeed", "true")
     .option("startingTimestamp", "2024-01-01")
     .load(path)
```

### CDF limitations
- Only available for tables with `delta.enableChangeDataFeed=true`
- Cannot read CDF before the feature was enabled
- Consumes extra storage for change data files
- Large UPDATE/DELETE operations produce large CDF files
